# 🥇 Gold Layer — Geographic Segments
**LushProtein Analytics | Medallion Architecture**

Reads from `medallion/gold/`, builds a customer-level geographic profile scoped to SG and MY, enriches with discount usage, top product category, and subscription flags, generates a country-level summary, and writes to `medallion/gold/`.

| Source Tables | Gold Tables | Key Transformations |
|---|---|---|
| `gold_customer_orders`, `gold_customer_profiles`, `gold_first_order_products`, `gold_subscription_behaviour` | `gold_geographic_segments`, `gold_geographic_summary` | Filter to SG/MY, pull customer-level metrics, compute discount usage rate, add top product category, merge subscription flags, aggregate country-level summary |

## 0. Setup & Paths

In [8]:
import pandas as pd
import numpy as np
import os
import re
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 80)

BASE       = Path.cwd()
SILVER_DIR = BASE / 'medallion' / 'silver'
GOLD_DIR   = BASE / 'medallion' / 'gold'
GOLD_DIR.mkdir(parents=True, exist_ok=True)

print(f"Base   : {BASE}")
print(f"Silver : {SILVER_DIR}")
print(f"Gold   : {GOLD_DIR}")

Base   : /Users/bennyteo/Desktop/Benny/MITB/Term 3/ISSS603 Applied Data Science for Customer Insights/Project/LushProtein_Project_Data_20260505
Silver : /Users/bennyteo/Desktop/Benny/MITB/Term 3/ISSS603 Applied Data Science for Customer Insights/Project/LushProtein_Project_Data_20260505/medallion/silver
Gold   : /Users/bennyteo/Desktop/Benny/MITB/Term 3/ISSS603 Applied Data Science for Customer Insights/Project/LushProtein_Project_Data_20260505/medallion/gold


---
## 1. Geographic Segments (`gold_customer_profiles` → `gold_geographic_segments`)

### Step 1 — Load data & filter to SG and MY customers

In [9]:
# Load data
co = pd.read_parquet(GOLD_DIR / 'gold_customer_orders.parquet')
cp = pd.read_parquet(GOLD_DIR / 'gold_customer_profiles.parquet')
fop = pd.read_parquet(GOLD_DIR / 'gold_first_order_products.parquet')

# Filter to SG and MY customers
# Use acquisition country from customer profiles
sg_my_customers = cp[cp['acquisition_country'].isin(['sg', 'my'])].copy()
print(f"SG customers: {(sg_my_customers['acquisition_country'] == 'sg').sum():,}")
print(f"MY customers: {(sg_my_customers['acquisition_country'] == 'my').sum():,}")

SG customers: 5,512
MY customers: 7,037


### Step 2 — Build customer-level geographic metrics

In [10]:
# Customer-level geographic metrics
geo = sg_my_customers[[
    'customer_id', 'acquisition_country', 'acquisition_channel',
    'first_order_date', 'last_order_date',
    'total_orders', 'total_revenue', 'avg_order_value',
    'total_discount_amt', 'is_discount_acquired', 'acquisition_discount_code',
    'repeat_purchase_90d', 'is_repeat_customer', 'days_to_second_order',
    'rfm_group', 'rfm_score', 'gender',
    'recency_days'
]].copy()

### Step 3 — Add discount usage rate across all orders

In [11]:
# Add order-level aggregations
# Discount usage rate across all orders (not just acquisition)
customer_discount_rate = (co[co['customer_id'].isin(geo['customer_id'])]
                           .groupby('customer_id')
                           .agg(
                               orders_with_discount=('discount_code', lambda x: x.notna().sum()),
                               total_orders_check  =('order_id', 'count')
                           ).reset_index())
customer_discount_rate['discount_usage_rate'] = (
    customer_discount_rate['orders_with_discount'] /
    customer_discount_rate['total_orders_check']
).round(3)

geo = geo.merge(customer_discount_rate[['customer_id', 'discount_usage_rate',
                                         'orders_with_discount']],
                on='customer_id', how='left')

### Step 4 — Add top product category from first order

In [12]:
# Add top product category from first order
top_category = (fop[fop['customer_id'].isin(geo['customer_id']) & fop['has_sku']]
                 .groupby('customer_id')['product_category']
                 .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
                 .reset_index()
                 .rename(columns={'product_category': 'top_product_category'}))

geo = geo.merge(top_category, on='customer_id', how='left')

### Step 5 — Add subscription flags from `gold_subscription_behaviour`

In [13]:
# Add subscription flag via customer ID bridge 
bridge = pd.read_parquet(SILVER_DIR / 'silver_customer_id_bridge.parquet')
sub = pd.read_parquet(GOLD_DIR / 'gold_subscription_behaviour.parquet')

# Map recharge customer_id to shopify customer_id
sub_with_shopify = sub.merge(bridge, on='customer_id', how='inner')
subscriber_shopify_ids = set(sub_with_shopify['shopify_customer_id'])
converted_shopify_ids = set(
    sub_with_shopify[sub_with_shopify['converted_to_subscription']]['shopify_customer_id']
)

geo['is_subscriber']             = geo['customer_id'].isin(subscriber_shopify_ids)
geo['converted_to_subscription'] = geo['customer_id'].isin(converted_shopify_ids)

### Step 6 — Build country-level summary & save

In [14]:
# Country-level summary for quick reference
country_summary = geo.groupby('acquisition_country').agg(
    total_customers      =('customer_id', 'count'),
    repeat_customers     =('is_repeat_customer', 'sum'),
    repeat_rate          =('is_repeat_customer', 'mean'),
    repeat_90d_rate      =('repeat_purchase_90d', 'mean'),
    avg_order_value      =('avg_order_value', 'mean'),
    median_order_value   =('avg_order_value', 'median'),
    avg_total_revenue    =('total_revenue', 'mean'),
    avg_orders           =('total_orders', 'mean'),
    discount_acquired_rate=('is_discount_acquired', 'mean'),
    avg_discount_usage   =('discount_usage_rate', 'mean'),
    subscriber_rate      =('is_subscriber', 'mean'),
).reset_index()

for col in ['repeat_rate', 'repeat_90d_rate', 'discount_acquired_rate',
            'avg_discount_usage', 'subscriber_rate']:
    country_summary[col] = country_summary[col].round(3)
for col in ['avg_order_value', 'median_order_value', 'avg_total_revenue', 'avg_orders']:
    country_summary[col] = country_summary[col].round(2)

print(f"\n=== Country Summary ===")
print(country_summary.T.to_string())

gold_geographic_segments = geo

print(f"\ngold_geographic_segments: {gold_geographic_segments.shape[0]:,} rows × {gold_geographic_segments.shape[1]} cols")

gold_geographic_segments.to_parquet(GOLD_DIR / 'gold_geographic_segments.parquet', index=False)
country_summary.to_parquet(GOLD_DIR / 'gold_geographic_summary.parquet', index=False)
print("\n✅ Saved: gold_geographic_segments.parquet")
print("✅ Saved: gold_geographic_summary.parquet")


=== Country Summary ===
                             0       1
acquisition_country         my      sg
total_customers           7037    5512
repeat_customers          2501    1867
repeat_rate              0.355   0.339
repeat_90d_rate          0.248   0.211
avg_order_value         207.99  107.28
median_order_value       122.4    62.1
avg_total_revenue       579.99  280.91
avg_orders                2.17    1.97
discount_acquired_rate    0.38   0.283
avg_discount_usage       0.394   0.297
subscriber_rate           0.03   0.056

gold_geographic_segments: 12,549 rows × 23 cols

✅ Saved: gold_geographic_segments.parquet
✅ Saved: gold_geographic_summary.parquet


#### Customer Base

MY: 7,037 customers (56.1%) vs SG: 5,512 (43.9%)

Malaysia has more customers driven by Lazada volume

#### Retention

MY repeat rate: 35.5% vs SG 33.9% — nearly identical, marginal MY advantage

MY 90-day repeat rate: 24.8% vs SG 21.1% — MY customers return slightly faster

Surprising finding — despite MY being predominantly marketplace (Lazada) and SG being predominantly DTC, retention rates are comparable. This challenges the assumption that DTC customers are inherently more loyal

#### Revenue

MY avg AOV: 207.99 vs SG 107.28 — MY is 1.9x higher

MY median AOV: 122.40 vs SG 62.10 — consistent even at median, ruling out outliers

MY avg lifetime revenue: 579.99 vs SG 280.91 — MY customers are worth 2x more in LTV

MY avg orders: 2.17 vs SG 1.97 — very similar frequency, so the LTV gap is driven by basket size not frequency

#### Discount Behaviour

MY discount acquisition rate: 38.0% vs SG 28.3% — MY customers more likely acquired via discount

MY avg discount usage: 39.4% vs SG 29.7% — MY customers also use discounts more across repeat orders

Despite higher discount dependency, MY customers still retain better and spend more — suggests discounts in MY are driving genuine incremental volume rather than just eroding margin

#### Subscriber Rate

MY: 3.0% vs SG: 5.6% — SG customers are nearly 2x more likely to convert to a subscription
Makes intuitive sense — DTC channel (dominant in SG) provides a smoother path to subscription conversion compared to Lazada (dominant in MY) where subscription isn't natively offered

This is a key finding for Q5 and Q8 — subscription conversion is a SG/DTC phenomenon, while MY growth is driven by one-time repeat purchases

#### Key takeaway for Q8
MY wins on volume, AOV, and LTV. SG wins on subscription conversion. The two markets have fundamentally different growth profiles — MY is a high-value repeat purchase market driven by marketplace, while SG is a subscription conversion market driven by DTC. These should be treated as separate strategic segments rather than a single market.